In [1]:
import gc
import os
import openslide
import albumentations as A
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import glob
import cv2

/usr/local/lib/python3.8/dist-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.4 (you have 1.4.18). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [2]:
def get_train_val_lists(csv_path):
    slide_list = pd.read_csv(csv_path)
    slide_list = slide_list.drop_duplicates(subset='slide_id', keep='first')
    aneuploid_list = slide_list.loc[slide_list['label'] == 'diploid', 'slide_id']
    diploid_list = slide_list.loc[slide_list['label'] == 'diploid', 'slide_id']


    # Split the lists into training and validation sets
    train_aneuploid, val_aneuploid = train_test_split(aneuploid_list, test_size=0.2, random_state=42)
    train_diploid, val_diploid = train_test_split(diploid_list, test_size=0.2, random_state=42)
    # Combine the lists 
    train_list = pd.concat([train_aneuploid, train_diploid], ignore_index=True)
    val_list = pd.concat([val_aneuploid, val_diploid], ignore_index=True)

    return train_list, val_list

def get_tif_files(base_path, csv_path):
    train_list, val_list = get_train_val_lists(csv_path)

    tif_files_train = []
    tif_files_val = []
    for slide_id in train_list:
        tif_files_train += glob.glob(f"{base_path}/{slide_id}*.tif")
    for slide_id in val_list:
        tif_files_val += glob.glob(f"{base_path}/{slide_id}*.tif")
    print(f"Number of training biopsies: {len(train_list)}")
    print(f"Number of validation biopsies: {len(val_list)}")

    return tif_files_train, tif_files_val

In [3]:
csv_path = "/rsrch5/home/trans_mol_path/cercan/BE_master/1.aneuploid/results/training_250131/sample_list_mda_12.csv"
base_path = "/rsrch5/home/trans_mol_path/cercan/BE_master/0.input/flow/roi/flow/MDA12_ROI_pyramid"


tif_files_train, tif_files_val = get_tif_files(base_path, csv_path)

Number of training biopsies: 474
Number of validation biopsies: 120


In [4]:
# generate 1024x1024 patches from the tif files
# read with openslide
# generate mask
# pass them to albumentations to crop to 448 x 448 patches
# test the mask to check if contains more than 60% tissue
# if passes save the region
# if the region cannot pass try for 5 times, fi it does not work, reduce the threshold 10% until is is completed


In [5]:
def read_slide(slide_path):
    slide = openslide.OpenSlide(slide_path)
    mask = generate_mask(slide)
    image = np.array(slide.read_region((0, 0), 0, slide.level_dimensions[0]).convert('RGB'))
    slide.close()
    return image, mask

def generate_mask(slide, level=0):
    level = slide.get_best_level_for_downsample(64)
    image_ds = np.array(slide.read_region((0, 0), level, slide.level_dimensions[level]).convert('RGB'))
    img_gray = cv2.cvtColor(image_ds, cv2.COLOR_RGB2GRAY)
    img_blur = cv2.GaussianBlur(img_gray,(7,7),0)

    _, mask_tissue = cv2.threshold(img_blur, 60, 255, cv2.THRESH_OTSU+cv2.THRESH_BINARY_INV)
    _, mask_ink = cv2.threshold(img_blur, 30, 255, cv2.THRESH_BINARY_INV)
    mask_foreground = cv2.subtract(mask_tissue, mask_ink)

    
    
    original_dimensions = slide.level_dimensions[0] 
    mask = cv2.resize(mask_foreground, (original_dimensions[0], original_dimensions[1]), interpolation=cv2.INTER_NEAREST)
    mask = np.array(mask)
    return mask

def check_tissue_percentage(mask, threshold=0.6):
    tissue_pixels = np.sum(mask > 0)
    total_pixels = mask.size
    return tissue_pixels / total_pixels >= threshold

In [6]:
def generate_and_save_patches(tif_files, output_dir, init_threshold=0.9, max_attempts=20):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    
    transform = A.Compose([
        A.RandomCrop(width=1024, height=1024, p=1)
    ])
    

    for tif_file in tif_files:
        slide_img, mask = read_slide(tif_file)
        # print(type(mask))
        # print(mask.shape)
            
        threshold = init_threshold
        total_regions = 0
        attempts = 0
        while attempts < max_attempts:
            if slide_img.shape[1] < 1024:
                transform_temp = A.Compose([
        A.RandomCrop(width=slide_img.shape[1], height=1024, p=1)
    ])
                region = transform_temp(image=slide_img, mask = mask)
            else:
                region = transform(image=slide_img, mask = mask)
            region_img = region['image']
            region_img =cv2.cvtColor(region_img, cv2.COLOR_BGR2RGB)
            region_mask = region['mask']
            if check_tissue_percentage(region_mask, threshold):
                patch_id = os.path.basename(tif_file).replace('.tif', f'_{total_regions}.png')
                patch_path = os.path.join(output_dir, patch_id)
                # print(patch_path)
                cv2.imwrite(patch_path, region_img)
                total_regions += 1
                if total_regions == 2:
                    break
                else:
                    attempts =0
                    threshold = init_threshold
                    
            attempts += 1

        if attempts == max_attempts:
            threshold -= 0.1
            attempts = 0

        gc.collect()




In [7]:
base_path = "/rsrch5/home/trans_mol_path/cercan/BE_master/0.input/flow/roi/flow/MDA12_ROI_pyramid"
output_path = "/rsrch5/home/trans_mol_path/cercan/BE_master/0.input/flow/roi/flow/MDA12_ROI_patches_finetuning_250217"

generate_and_save_patches(tif_files_train, os.path.join(output_path,"train"))
generate_and_save_patches(tif_files_val, os.path.join(output_path,"val"))

